# 03 — Training, evaluation và inference
Mặc định chạy CPU Kaggle. Cần labeling/pool-manifest.json và judgments.csv đã chấm. Stage train chỉ dùng train/validation. Chỉ mở test sau khi khóa lựa chọn; cần test-adjudicated.csv và previous-run chứa selection-lock cùng artifact train.

In [ ]:
DENSE_DEVICE = 'cpu'
EXTRAS = '[kaggle]'
from pathlib import Path
import os, sys, subprocess, zipfile, shutil, importlib.util
if not Path('/kaggle/input').exists():
    raise RuntimeError('Notebook này chỉ chạy trên Kaggle, không train trên máy cá nhân.')
archives = list(Path('/kaggle/input').rglob('source.zip'))
source = Path('/kaggle/working/fashion-reco/source')
source.mkdir(parents=True, exist_ok=True)
if len(archives) == 1:
    workspace = archives[0].parent
    with zipfile.ZipFile(archives[0]) as archive:
        for member in archive.infolist():
            if not (source / member.filename).resolve().is_relative_to(source.resolve()):
                raise ValueError('Unsafe archive path')
        archive.extractall(source)
elif not archives:
    sources = [p.parent for p in Path('/kaggle/input').rglob('pyproject.toml') if (p.parent/'src/fashion_reco').is_dir()]
    if len(sources) != 1:
        raise RuntimeError('Thêm đúng một workspace có source.zip hoặc thư mục source/.')
    workspace = sources[0].parent
    shutil.copytree(sources[0], source, dirs_exist_ok=True)
else:
    raise RuntimeError('Có nhiều file source.zip trong input.')
if EXTRAS:
    # Import our source directly; never invoke a package build on Kaggle.
    requirements = {'numpy':'numpy>=1.26,<3', 'sklearn':'scikit-learn>=1.4,<2',
                    'yaml':'PyYAML>=6,<7', 'joblib':'joblib>=1.3,<2',
                    'rank_bm25':'rank-bm25>=0.2.2,<1', 'torch':'torch',
                    'sentence_transformers':'sentence-transformers>=3,<6',
                    'lightgbm':'lightgbm>=4,<5', 'pandas':'pandas'}
    wheels = list(workspace.rglob('rank_bm25-*.whl'))
    if importlib.util.find_spec('rank_bm25') is None and len(wheels)==1:
        subprocess.run([sys.executable,'-m','pip','install','--no-index','--no-deps',str(wheels[0])],check=True)
    missing = [value for name,value in requirements.items() if importlib.util.find_spec(name) is None]
    if missing:
        subprocess.run([sys.executable,'-m','pip','install','--timeout','25','--retries','1',*missing],check=True)
    manifest_path = workspace/'offline-model-manifest.json'
    if manifest_path.exists():
        import json
        spec = importlib.util.spec_from_file_location('fashion_kaggle_runtime',source/'kaggle/run_experiment.py')
        runtime = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(runtime)
        runtime.OFFLINE_MODELS = json.loads(manifest_path.read_text(encoding='utf-8'))
        runtime.prepare_offline_models()
sys.path.insert(0, str(source / 'src'))
output = Path('/kaggle/working/fashion-reco/output')
if (workspace / 'previous-run').exists():
    shutil.copytree(workspace / 'previous-run', output, dirs_exist_ok=True)
os.environ['FASHION_CATALOG'] = str(workspace / 'catalog.json')
os.environ['FASHION_OUTPUT'] = str(output)
config_path = source / 'configs/experiment.yaml'
if EXTRAS:
    import torch, yaml
    device = globals().get('DENSE_DEVICE','cpu')
    if device not in ('cpu','cuda'):
        raise ValueError('Choose cpu or cuda explicitly')
    config = yaml.safe_load(config_path.read_text(encoding='utf-8'))
    config['dense'].update(device=device,batch_size=8 if device=='cpu' else 16)
    if device == 'cpu':
        torch.set_num_threads(4)
    elif not torch.cuda.is_available():
        raise RuntimeError('CUDA unavailable. Choose CPU or enable an actual GPU on Kaggle.')
    config_path.write_text(yaml.safe_dump(config,sort_keys=False),encoding='utf-8')
    print('Selected cloud device:',device)


In [ ]:
STAGE = 'train'  # Chỉ đổi sang test sau khi chốt mô hình.
from fashion_reco.workflow import run_stage
labeling_dir = workspace / 'labeling'
if not labeling_dir.exists():
    raise RuntimeError('Chưa có nhãn thật. Hoàn thành notebook 02 và upload labeling trước.')
result = run_stage(config_path, STAGE, labeling_dir=labeling_dir)
print(result)

In [ ]:
import pandas as pd, json
summary = result['validation'] if STAGE == 'train' else result['test_ranking']
display(pd.DataFrame(summary).T)
print('Lựa chọn khóa bằng validation:', result['selected_method'])

## Xem bộ sản phẩm kèm ảnh
Sau train: xem bộ validation của hai phương án đứng đầu. Sau test: xem kết quả của phương án đã chọn. Ảnh là URL nguồn và có thể lỗi. Chưa có bảo đảm về tính tương thích thẩm mỹ.

In [ ]:
from IPython.display import HTML, display
from html import escape
bundle_file = output / ('validation-bundles.json' if STAGE == 'train' else 'test-bundles.json')
if bundle_file.exists():
    bundles = json.loads(bundle_file.read_text(encoding='utf-8'))
    examples = [b for b in bundles if b['method'] == result['selected_method'] and b['mode'] == 'available_only'][:4]
    for bundle in examples:
        html = f"<h3>{escape(bundle['query_id'])} — {escape(bundle['status'])}</h3><p>Tổng: {bundle['total_price']:,} / {bundle['budget']:,} VND</p>"
        for p in bundle['items']:
            image = p['image_url'] if p['image_url'].startswith('https://') else ''
            html += f"<div style='display:inline-block;vertical-align:top;width:220px;padding:12px'><img src='{escape(image, quote=True)}' width='160'><p>{escape(p['name'])}</p><p>{p['price']:,} VND</p></div>"
        display(HTML(html))
else:
    print('Chưa có kết quả bộ sản phẩm.')

## Bàn giao
Lưu model, embedding, selection-lock, environment và build timings. Nhóm chấm validation/test-bundle-human-review.csv về đáp ứng nhu cầu và tương thích. Không chọn lại model bằng test. Peak memory là toàn process; muốn đo từng model cần process riêng.